# 03 · Select samples and attach patient metadata

**Run order:** 02 make facility CSVs → **03 (this notebook)** → 04 batch PCA + ComBat-seq. Kernel: **Python 3**.

**What this does.** Reads the four facility CSVs from 02 (patients + healthy controls), then:
whole blood only → library size ≥ 1M → patient metadata join (age, sex...) → pre/post treatment → **drops post-treatment** → eligible set → **one sample per patient** (explicit PRE first, then **earliest collection date**) → gene lengths → RPKM / TPM → CSVs.

**Same patient at two facilities?** Not a problem. Every file is read first and the patient identity (`patient_key`) is built across all of them, so the one-per-patient choice is made *across* facilities, not within each. The "eligible" set keeps all samples (a patient seen at two facilities has two samples, both usable for differential expression with `patient_key` as a block).

**Reads:** `rnaseq_out_v2/facility_csv/*.csv` (from 02), `MetadataPatients_NoPHI.xlsx`. **Writes:** `rnaseq_out_v2/` (all CSV).
Birth date is used only to compute age; outputs contain no birth dates and no raw collection dates.

| Output | Use |
|---|---|
| `rnaseq_samples_eligible.csv` + `rnaseq_counts_raw_eligible.csv` | **all eligible samples** → notebook 04, differential expression (batch term + patient block) |
| `rnaseq_samples.csv` + `rnaseq_counts_raw.csv` | **one per patient** → WGCNA, anything needing independent samples |
| `rnaseq_rpkm.csv` / `rnaseq_tpm.csv` | display values for the one-per-patient cohort |
| `rnaseq_samples_ALL_with_flags.csv` | every sample with the reason it was kept or dropped (`status`) |
| `batch_by_diagnosis_eligible.csv` | which batches can be separated from diagnosis |

---
# Part 1 · Configuration
All paths and rules are here.

In [ ]:
import os, re, glob, gzip, shutil, datetime, urllib.request
import numpy as np, pandas as pd

MNT = "/mnt/file-systems/"
RESULTS   = "/home/jovyan/session_data/endotype_results"
OUT_DIR   = RESULTS + "/rnaseq_out_v2"
RNA_FILES = sorted(f for f in glob.glob(OUT_DIR + "/facility_csv/*.csv")
                   if not os.path.basename(f).startswith("facility_csv_summary"))   # written by notebook 02
METADATA_PATH = MNT + "MetadataPatients_NoPHI.xlsx"
METADATA_EXTRA = []        # e.g. a healthy-volunteer sheet with the same headers
CACHE_DIR = "/home/jovyan/rnaseq_cache"
LEGACY_LENGTHS = MNT + "rnaseq_out/gene_lengths.csv"   # reused if present, so no GTF download

KEEP_ONE_PER_PATIENT = "earliest"            # or "deepest" (most total counts)
REQUIRE_PRE_FOR_DX = ""                      # "" = only drop post-treatment. r"CANDLE|SAVI" = also demand an explicit PRE label
MIN_LIBRARY = 1e6                            # samples with fewer total counts are dropped (a failed library)
BLOOD_VALUES = {"blood", "whole blood", "wholeblood", "wb"}   # exact match; PBMC never passes
GTF_PATH = ""                                # blank = auto-find, then try download
GTF_URL = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"
           "release_19/gencode.v19.annotation.gtf.gz")
DX_MAP = {"hc whole blood": "HC",
          "relative-hc": "HC",
          "undifferentiated interferonopathy": "Undifferentiated Interferonopathy"}
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), f"{OUT_DIR} is not writable"
assert RNA_FILES, f"no facility CSVs in {OUT_DIR}/facility_csv: run notebook 02 first"
print("facility files:", [os.path.basename(f) for f in RNA_FILES])

---
# Part 2 · Helpers
`normalize_id` makes `O-6033`, `o6033` and `O6033` compare equal and maps RNA's `CHLD0047` to the metadata's `LD0047`. `read_workbook_cached` returns every sheet of an Excel file (or the one CSV).

In [ ]:
def normalize_id(x):
    s = "" if x is None else str(x).strip()
    if s.lower() in ("nan", "none", "nat", ""): return ""
    s = re.sub(r"\.0$", "", s)
    s = s.upper().replace("-", "").replace(" ", "").replace("_", "")
    if s.startswith("CHLD"):                 # CHLD0047 (RNA files) == LD0047 (metadata)
        s = s[2:]
    return s

def norm_colname(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())

def _safe(s): return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")

DATE_MIN, DATE_MAX = pd.Timestamp("2008-01-01"), pd.Timestamp.today().normalize()

def read_workbook_cached(path):
    """Return {sheet name: DataFrame of text} for EVERY sheet of an .xlsx (or the one csv)."""
    stem = os.path.splitext(os.path.basename(path))[0]
    if path.lower().endswith(".csv"):
        return {stem: pd.read_csv(path, dtype=object, keep_default_na=False, low_memory=False)}
    xl = pd.ExcelFile(path)
    out = {}
    for sh in xl.sheet_names:
        csv_path = os.path.join(CACHE_DIR, f"{stem}__{_safe(sh)}.csv")
        if (not os.path.exists(csv_path)) or os.path.getmtime(csv_path) < os.path.getmtime(path):
            print(f"converting {os.path.basename(path)} [{sh}] -> csv (one time) ...")
            xl.parse(sh, dtype=object).to_csv(csv_path, index=False)
        out[sh] = pd.read_csv(csv_path, dtype=object, keep_default_na=False, low_memory=False)
    return out

---
# Part 3 · Patient metadata
The sheet with the most *enroll / diagnos / initial* columns is chosen automatically. Initials map to an ID **only where they are unique**: a wrong patient is worse than a blank one.

In [ ]:
sheets = pd.read_excel(METADATA_PATH, sheet_name=None, dtype=str)
md = max(sheets.values(), key=lambda d: sum(bool(re.search(r"enroll|diagnos|initial", str(c), re.I))
                                            for c in d.columns))
md.columns = [str(c).strip() for c in md.columns]
md = md.dropna(how="all").reset_index(drop=True)

for p in METADATA_EXTRA:                     # main sheet first, so it wins on duplicate IDs
    ex = (pd.read_excel(p, dtype=str) if p.lower().endswith((".xlsx", ".xls"))
          else pd.read_csv(p, dtype=str))
    ex.columns = [str(c).strip() for c in ex.columns]
    nm = {norm_colname(c): c for c in md.columns}
    ex = ex.rename(columns={c: nm[norm_colname(c)] for c in ex.columns if norm_colname(c) in nm})
    md = pd.concat([md, ex.dropna(how="all").reindex(columns=md.columns)], ignore_index=True)
    print(f"extra metadata: {p} (+{len(ex)} rows)")

def fc(*names):
    nm = {norm_colname(c): c for c in md.columns}
    return next((nm[norm_colname(n)] for n in names if norm_colname(n) in nm), None)

COL = {"enroll": fc("Enrollment ID"), "alt": fc("Alternate Enrollment ID"),
       "initials": fc("Initials"), "dob": fc("Birth Date", "DOB"), "sex": fc("Sex", "Gender"),
       "race": fc("Race"), "ethnicity": fc("Ethnicity"), "status": fc("Study Status"),
       "dx1": fc("Diagnosis1"), "dx2": fc("Diagnosis2"), "note": fc("Note", "Notes")}
print("metadata:", len(md), "rows; columns not found:", [k for k, v in COL.items() if v is None])

meta_lookup, by_ini = {}, {}
for i, r in md.iterrows():
    for kc in (COL["enroll"], COL["alt"]):
        k = normalize_id(r.get(kc)) if kc else ""
        if k: meta_lookup.setdefault(k, i)
    ini = str(r.get(COL["initials"]) or "").strip().upper() if COL["initials"] else ""
    eid = normalize_id(r.get(COL["enroll"])) if COL["enroll"] else ""
    if ini and eid: by_ini.setdefault(ini, set()).add(eid)
AUTO_INITIALS_MAP = {k: next(iter(v)) for k, v in by_ini.items() if len(v) == 1}

def meta_value(i, key):
    c = COL.get(key)
    if i < 0 or not c: return ""
    v = md.at[i, c]
    return "" if pd.isna(v) else str(v).strip()

---
# Part 4 · Load the facility CSVs
In each CSV, rows labelled `Raw_Count` are gene counts; the other rows are sample information (diagnosis, treatment, facility...). The loader splits them and turns each sample into one row. Each CSV is one "unit" (one per facility). Units are joined on `Gene_ID`; the notebook prints whether they list the same genes in the same order.

**Compare the per-unit sample counts with the table printed by notebook 02** (patients + healthy controls per facility). Samples with blank counts are flagged, not deleted.

In [ ]:
REQUIRED = {"Gene_ID", "Gene_Name", "Stat"}

def load_unit(df, source_file, sheet):
    raw = df.copy()
    raw.columns = [str(c).strip() for c in raw.columns]
    raw = raw.drop(columns=[c for c in raw.columns if c.startswith("Unnamed: 0")])   # the row-number column
    if not REQUIRED <= set(raw.columns):
        return None
    samples = [c for c in raw.columns if c not in REQUIRED]
    is_count = raw["Stat"].astype(str).str.strip().eq("Raw_Count")
    meta = raw.loc[~is_count, ["Stat"] + samples].set_index("Stat").T
    meta.columns = [str(c).strip() for c in meta.columns]
    meta.index.name = "sample_id"
    meta = meta.reset_index()
    meta["source_file"], meta["source_sheet"] = source_file, sheet
    cnt = raw.loc[is_count, ["Gene_ID", "Gene_Name"] + samples].copy()
    cnt[samples] = cnt[samples].apply(pd.to_numeric, errors="coerce")
    for s in samples:
        if cnt[s].isna().all():
            print(f"WARNING [{sheet}]: no numeric counts in sample {s}")
    return meta, cnt

units, metas, cnts = [], [], []
for p in RNA_FILES:
    for sheet, df in read_workbook_cached(p).items():
        res = load_unit(df, os.path.basename(p), sheet)
        if res is None:
            print(f"skipped {os.path.basename(p)} [{sheet}]: no Gene_ID / Gene_Name / Stat columns")
            continue
        m, c = res
        units.append({"file": os.path.basename(p)[:40], "sheet": sheet,
                      "samples": c.shape[1] - 2, "genes": len(c)})
        metas.append(m); cnts.append(c)
units_df = pd.DataFrame(units)
print(units_df.to_string(index=False))

ids0 = cnts[0]["Gene_ID"].tolist()
same_order = [c["Gene_ID"].tolist() == ids0 for c in cnts]
print(f"\nunits listing the same genes in the same order as the first unit: {sum(same_order)} of {len(cnts)}")

### 4.2 Combine into one count matrix and one sample table
Field names are standardised (`Enrollment ID` / `EnrollmentID` → one name); placeholder text becomes empty. `total_counts` is the library size. **`batch`** = the facility file the sample is in (so `BMC-HGSC` / `BCM-HGSC` spellings can never split one facility into two batches); for NIAMS it is facility + sequencing cohort (`Batch1`, `Batch2`, `unlabelled`). Samples with no counts, blank count cells, or fewer than `MIN_LIBRARY` counts, are flagged and excluded later with the reason recorded.

In [ ]:
gene_names = cnts[0].drop_duplicates("Gene_ID").set_index("Gene_ID")["Gene_Name"]
mats = [c.drop(columns="Gene_Name").groupby("Gene_ID").sum(min_count=1) for c in cnts]
counts = pd.concat(mats, axis=1, join="inner")
print(f"genes in every unit: {len(counts)}  (per unit: {[len(m) for m in mats]})")
dup = counts.columns[counts.columns.duplicated()]
if len(dup): raise ValueError(f"sample names repeated across units: {list(dup)[:5]}")

rn = {"enrollmentid": "enrollment_id_rna", "metadatainitials": "initials_rna",
      "samplesource": "sample_source", "diagnosis": "diagnosis_rna", "treatment": "treatment",
      "sequencingfacility": "sequencing_facility", "sequencingcohort": "sequencing_cohort",
      "gender": "gender_rna", "rnasamplecollectiondate": "rna_collection_date"}
rmeta = pd.concat(metas, ignore_index=True)
rmeta = rmeta.rename(columns=lambda c: rn.get(norm_colname(c), c))
for c in rn.values():
    if c not in rmeta.columns: rmeta[c] = ""
rmeta = rmeta.fillna("").replace({"-": "", "NaN": "", "nan": ""})
rmeta["total_counts"] = rmeta["sample_id"].map(counts.sum(min_count=1))
rmeta["n_blank_genes"] = rmeta["sample_id"].map(counts.isna().sum())
rmeta["has_counts"] = (rmeta["total_counts"].fillna(0) > 0) & (rmeta["n_blank_genes"] == 0)   # blank cells are never treated as 0
rmeta["lib_ok"] = rmeta["total_counts"].fillna(0) >= MIN_LIBRARY
print(f"\nsamples with no counts at all (excluded below): {int((rmeta['total_counts'].fillna(0) <= 0).sum())}")
print(f"samples with some blank counts (excluded below): {int(((rmeta['n_blank_genes'] > 0) & (rmeta['total_counts'].fillna(0) > 0)).sum())}")
print(f"samples below the minimum library size of {MIN_LIBRARY:,.0f} (excluded below): "
      f"{int((rmeta['has_counts'] & ~rmeta['lib_ok']).sum())}",
      rmeta.loc[rmeta["has_counts"] & ~rmeta["lib_ok"], "sample_id"].tolist())
rmeta["dx_clean"] = rmeta["diagnosis_rna"].map(
    lambda s: DX_MAP.get(str(s).strip().lower(), str(s).strip()))

# ---- batch = facility (NIAMS split by sequencing cohort) --------------------
_blank = ["", "NA", "nan", "-"]
fac = rmeta["source_sheet"].astype(str).str.strip()      # the file = the facility (notebook 02 put each healthy control in its own facility's file)
_k = lambda x: re.sub(r"[^A-Z0-9]", "", str(x).upper()).replace("BMCHGSC", "BCMHGSC")
written = rmeta["sequencing_facility"].astype(str).str.strip()
disagree = (~written.isin(_blank)) & (written.map(_k) != fac.map(_k))
print(f"samples whose own Sequencing_Facility line disagrees with the file they are in: {int(disagree.sum())}")
coh = rmeta["sequencing_cohort"].astype(str).str.strip()
coh = coh.where(~coh.isin(_blank), "unlabelled")
rmeta["facility"] = fac
rmeta["batch"] = np.where(fac.str.upper().eq("NIAMS"), "NIAMS-" + coh, fac)
print("\nsamples per batch (all samples read):")
print(rmeta.groupby(["source_sheet", "batch"]).size().rename("n").reset_index().to_string(index=False))

### Checkpoint 4: what did we load?
Left: samples per diagnosis. Right: sequencing depth per sample (log scale).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
rmeta["dx_clean"].replace("", "(blank)").value_counts().head(12).plot.barh(ax=ax[0], color="#4C78A8")
ax[0].set_title(f"Samples per diagnosis (n={len(rmeta)})"); ax[0].invert_yaxis()
ax[1].hist(np.log10(rmeta["total_counts"].fillna(0).clip(lower=1)), bins=40, color="#76B7B2")
ax[1].set_title("Library size per sample"); ax[1].set_xlabel("log10(total counts)")
plt.tight_layout(); plt.show()

---
# Part 5 · Whole blood only
`sample_source` must be *exactly* one of `BLOOD_VALUES` (so `B cells`, `Neutrophils`, `skin` are out). Even if labelled blood, a sample whose name/treatment/cohort text names a cell fraction is excluded and printed.

In [ ]:
src = rmeta["sample_source"].astype(str).str.strip().str.lower()
print("\nsample source values (per file):")
print(pd.crosstab(rmeta["source_sheet"].str[:30], src).to_string())
NOT_WHOLE_BLOOD = re.compile(r"pbmc|mononuclear|monocyte|neutrophil|lymphocyte|"
                             r"t[\s_-]?cell|b[\s_-]?cell|nk[\s_-]?cell|cd\d|sorted|isolat", re.I)
blob = (rmeta["sample_id"].astype(str) + " " + src + " " + rmeta["treatment"].astype(str)
        + " " + rmeta["sequencing_cohort"].astype(str))
sneaky = src.isin(BLOOD_VALUES) & blob.str.contains(NOT_WHOLE_BLOOD)
print(f"\nlabelled blood but name suggests a cell fraction: {int(sneaky.sum())}")
if sneaky.any():
    print(rmeta.loc[sneaky, ["sample_id", "sample_source"]].head(20).to_string())
rmeta["is_blood"] = src.isin(BLOOD_VALUES) & ~sneaky

---
# Part 6 · Dates
A best-available draw date per sample, used only for age and for choosing the *earliest* sample. A real `rna_collection_date` is preferred; if missing, a date is read from the sample name (e.g. `121003_121016`), which is only a rough proxy (a sequencing-batch date). Dates outside 2008-today are rejected as typos. Dates are never printed or saved.

In [ ]:
def date_from_name(s):
    m = re.search(r"(?<!\d)(\d{6})_\d{6}(?!\d)", s) or re.search(r"(?<!\d)(\d{6})(?!\d)", s)
    if not m: return pd.NaT
    try: d = pd.to_datetime(m.group(1), format="%y%m%d")
    except ValueError: return pd.NaT
    return d if DATE_MIN <= d <= DATE_MAX else pd.NaT

coll = pd.to_datetime(rmeta["rna_collection_date"], errors="coerce")
coll = coll.where((coll >= DATE_MIN) & (coll <= DATE_MAX))
from_name = rmeta["sample_id"].map(date_from_name)
rmeta["_date"] = coll.fillna(from_name)
rmeta["age_date_source"] = np.where(coll.notna(), "rna_collection_date",
                            np.where(from_name.notna(), "sample_name (approx)", "none"))
print(rmeta["age_date_source"].value_counts().to_string())

---
# Part 7 · Join patient metadata
A ladder, most trusted first: exact **Enrollment ID**; the HC fix (`1126` is `R1126` in the metadata); **initials, only when safe** (blank/initials-derived RNA ID, unique initials, diagnosis compatible). A real-looking ID the metadata lacks is **never guessed** (`id_not_in_metadata`). `demographics_trusted` is true only for the two strong ID methods.

In [ ]:
HEALTHY = re.compile(r"^hc$|health|control", re.I)
def match_meta(r):
    k = normalize_id(r["enrollment_id_rna"])
    ini = str(r["initials_rna"]).strip().upper()
    ini_n = normalize_id(ini)
    if k and k in meta_lookup: return meta_lookup[k], "enrollment_id"
    if k.isdigit() and ("R" + k) in meta_lookup:           # HC '1126' is 'R1126' in the metadata
        return meta_lookup["R" + k], "R_prefix_id"
    # only fall back to initials when the RNA ID is blank or is itself initials-derived (HC-MF, BG-SAF)
    if not (k == "" or (ini_n and k.endswith(ini_n))):
        return -1, "id_not_in_metadata"                    # a real-looking ID that metadata lacks: don't guess
    eid = AUTO_INITIALS_MAP.get(ini)                       # unique initials only
    if eid and eid in meta_lookup:
        i = meta_lookup[eid]
        dx_md, dx_rna = meta_value(i, "dx1"), str(r["diagnosis_rna"])
        same = (norm_colname(dx_md) == norm_colname(dx_rna)
                or (HEALTHY.search(dx_md) and HEALTHY.search(dx_rna)))
        compat = (re.search(r"interferon|aicardi|\bags\b", dx_md, re.I)
                  and re.search(r"\bags\b|aicardi", dx_rna, re.I))
        if dx_md == "" or same: return i, "initials"
        if compat: return i, "initials_dx_compat"
        return -1, "initials_conflict"
    return -1, "none"

mm = rmeta.apply(lambda r: pd.Series(match_meta(r), index=["meta_row", "meta_join"]), axis=1)
rmeta = pd.concat([rmeta, mm], axis=1)
extra_cols = {}
for f, name in [("sex", "sex"), ("race", "race"), ("ethnicity", "ethnicity"),
                ("status", "study_status"), ("dx1", "diagnosis1"), ("dx2", "diagnosis2"),
                ("note", "metadata_note"), ("dob", "_dob"), ("enroll", "_md_enroll")]:
    extra_cols[name] = rmeta["meta_row"].map(lambda i, f=f: meta_value(i, f))
rmeta = pd.concat([rmeta, pd.DataFrame(extra_cols)], axis=1)
rmeta["demographics_trusted"] = rmeta["meta_join"].isin(["enrollment_id", "R_prefix_id"])
print("\nmetadata join:"); print(rmeta["meta_join"].value_counts().to_string())

dob = pd.to_datetime(rmeta["_dob"], errors="coerce")
rmeta["age_yrs"] = ((rmeta["_date"] - dob).dt.days / 365.25).round(1)
rmeta.loc[(rmeta["age_yrs"] < 0) | (rmeta["age_yrs"] > 120), "age_yrs"] = np.nan

---
# Part 8 · Treatment status
Each sample is labelled `pre`, `post` or `unlabelled`. The treatment text is free-form: a pattern recognises "pre" wording and **any other non-blank text is treated as post-treatment** (the cautious default: a JAK inhibitor can suppress an interferon signature so a post-treatment sample looks healthy). **Post-treatment samples are dropped in Part 9.** Blank treatment (all healthy controls) is kept.

In [ ]:
PRE_TX = re.compile(r"pre|baseline|bsl|untreat|naive|none|before|screen|off[\s-]?(?:tx|treat)|^v?0$|^day\s*0$", re.I)
tx = rmeta["treatment"].astype(str).str.strip()
is_pre = (tx != "") & tx.str.contains(PRE_TX)
rmeta["is_post_tx"] = (tx != "") & ~is_pre
rmeta["tx_status"] = np.where(rmeta["is_post_tx"], "post", np.where(is_pre, "pre", "unlabelled"))
print("\ntreatment status (per file):")
print(pd.crosstab(rmeta["source_sheet"].str[:30], rmeta["tx_status"]).to_string())

### Checkpoint 8: which treatment strings were called what?
This regex decides who is in the cohort. Scan for anything misclassified and fix `PRE_TX` if so.

In [ ]:
display(rmeta.assign(treatment=rmeta["treatment"].replace("", "(blank)"))
        .groupby(["tx_status", "treatment"]).size().rename("n")
        .reset_index().sort_values(["tx_status", "n"], ascending=[True, False]).head(40))

---
# Part 9 · Eligible set, one per patient, and can batch be separated from diagnosis?
**Eligible** = whole blood, counts and library size OK, **not post-treatment**, a patient identity exists (and an explicit PRE label for diagnoses in `REQUIRE_PRE_FOR_DX`, if you set it).
**One per patient** (across all facilities): explicit PRE beats unlabelled, then **earliest date** (or deepest library if `"deepest"`), library size breaks ties.

**The batch × diagnosis table is the key output.** A batch holding one diagnosis cannot be separated from it by any model term. Healthy controls are appended per facility, so each facility has its own within-batch reference where HC exist.

In [ ]:
rmeta["patient_key"] = [normalize_id(a) or normalize_id(b) or normalize_id(c)
    for a, b, c in zip(rmeta["_md_enroll"], rmeta["enrollment_id_rna"], rmeta["initials_rna"])]
need_pre = (rmeta["dx_clean"].astype(str).str.contains(REQUIRE_PRE_FOR_DX, case=False, regex=True)
            if REQUIRE_PRE_FOR_DX else pd.Series(False, index=rmeta.index))
base = (rmeta["is_blood"] & rmeta["has_counts"] & rmeta["lib_ok"] & ~rmeta["is_post_tx"]
        & (rmeta["patient_key"] != ""))
print(f"excluded for lacking a PRE label ({REQUIRE_PRE_FOR_DX or 'rule off'}):", int((base & need_pre & ~is_pre).sum()))
elig = base & (~need_pre | is_pre)
rmeta["eligible"] = elig
b = rmeta[elig].copy()
b["_pre_rank"] = (b["tx_status"] != "pre").astype(int)       # explicit PRE beats unlabelled
if KEEP_ONE_PER_PATIENT == "earliest":
    b = b.sort_values(["patient_key", "_pre_rank", "_date", "total_counts"],
                      ascending=[True, True, True, False], na_position="last")
else:
    b = b.sort_values(["patient_key", "_pre_rank", "total_counts"], ascending=[True, True, False])
rmeta["kept"] = rmeta["sample_id"].isin(b.drop_duplicates("patient_key")["sample_id"])
rmeta["one_per_patient"] = rmeta["kept"]
final_meta = rmeta[rmeta["kept"]].copy()           # one per patient
elig_meta = rmeta[rmeta["eligible"]].copy()        # all eligible samples

rmeta["status"] = np.select(
    [~rmeta["has_counts"], ~rmeta["lib_ok"], ~rmeta["is_blood"], rmeta["is_post_tx"],
     rmeta["patient_key"] == "", need_pre & ~is_pre, ~rmeta["kept"]],
    ["no counts / blank counts in file", "library below minimum", "not whole blood", "post-treatment", "no patient ID",
     "no PRE label (rule on)", "eligible, not the chosen sample of this patient"],
    default="kept")
rmeta.loc[rmeta["eligible"] & ~rmeta["kept"], "status"] = "eligible, not the chosen sample of this patient"

fac_per_pt = elig_meta.groupby("patient_key")["facility"].nunique()
print(f"\nsamples: {len(rmeta)} -> {int(rmeta['is_blood'].sum())} blood "
      f"-> {int(elig.sum())} eligible ({elig_meta['patient_key'].nunique()} patients) "
      f"-> {len(final_meta)} one-per-patient")
print(f"patients with samples at 2+ facilities: {int((fac_per_pt > 1).sum())}; "
      f"patients with 2+ eligible samples: {int((elig_meta['patient_key'].value_counts() > 1).sum())}")
print("age known (one per patient):", int(final_meta["age_yrs"].notna().sum()), "of", len(final_meta))
for title, col in [("treatment status", "tx_status"), ("metadata join", "meta_join"),
                   ("age date source", "age_date_source")]:
    print(f"\nkept (one per patient) by {title}:")
    print(pd.crosstab(final_meta["dx_clean"].replace("", "(blank)"), final_meta[col]).to_string())

def batch_table(df, title):
    t = pd.crosstab(df["batch"], df["dx_clean"].replace("", "(blank)"))
    n_dx = (t > 0).sum(axis=1)
    t = t.assign(n_diagnoses=n_dx)
    print(f"\n{title}")
    print(t.to_string())
    one = list(t.index[n_dx == 1])
    if one:
        print("batches holding a single diagnosis (cannot be separated from it by a batch term):", one)
    return t

bt_elig = batch_table(elig_meta, "BATCH x DIAGNOSIS, all eligible samples")
bt_kept = batch_table(final_meta, "BATCH x DIAGNOSIS, one per patient")

ini_ct = final_meta.assign(ini=final_meta["initials_rna"].str.upper()).groupby("ini")["patient_key"].agg(list)
print("\nsame initials under more than one patient key (possible duplicate person):")
print(ini_ct[ini_ct.str.len() > 1].to_string() if (ini_ct.str.len() > 1).any() else "  none")

### Checkpoint 9
Left: the fate of every sample. Right: eligible samples by batch, coloured by diagnosis (a single-colour bar = a batch that cannot be separated from that diagnosis).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
rmeta["status"].value_counts().plot.barh(ax=ax[0], color="#4C78A8")
ax[0].invert_yaxis(); ax[0].set_title(f"Fate of all {len(rmeta)} samples")
bt_elig.drop(columns="n_diagnoses").plot.barh(stacked=True, ax=ax[1])
ax[1].invert_yaxis(); ax[1].set_title("Eligible samples by batch and diagnosis"); ax[1].set_ylabel("")
ax[1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

---
# Part 10 · Gene lengths → RPKM and TPM (display values)
Raw counts are the analysis input (notebook 04 and differential expression use them with batch as a model term). RPKM = counts ÷ length(kb) ÷ library(millions); TPM = (counts ÷ length) ÷ sum × 10⁶ (every sample sums to 1,000,000). Gene length = non-overlapping exon union per gene from a GENCODE GTF: a cached `gene_lengths.csv` is used first, then any `.gtf` on the mounts, then a GENCODE v19 download. The match rate is printed. Raw counts are written *before* this step can fail.

In [ ]:
def gtf_gene_lengths(path):
    op = gzip.open if path.endswith(".gz") else open
    rows = []
    with op(path, "rt") as fh:
        for line in fh:
            if line[0] == "#": continue
            f = line.split("\t")
            if len(f) > 8 and f[2] == "exon":
                rows.append((re.search(r'gene_id "([^"]+)"', f[8]).group(1), int(f[3]), int(f[4])))
    ex = pd.DataFrame(rows, columns=["g", "s", "e"]).sort_values(["g", "s"])
    prev = ex.groupby("g")["e"].cummax().groupby(ex["g"]).shift().fillna(0)
    return ((ex["e"] - np.maximum(ex["s"] - 1, prev)).clip(lower=0)).groupby(ex["g"]).sum()

len_csv = os.path.join(OUT_DIR, "gene_lengths.csv")
if not os.path.exists(len_csv) and os.path.exists(LEGACY_LENGTHS):
    shutil.copy2(LEGACY_LENGTHS, len_csv)
    print("reused gene lengths from the previous run:", LEGACY_LENGTHS)
lengths = pd.read_csv(len_csv, index_col=0).iloc[:, 0] if os.path.exists(len_csv) else None
if lengths is None:
    gtf = GTF_PATH or next(iter(sum((glob.glob(f"{r}/**/*.gtf*", recursive=True)
            for r in ["/mnt/file-systems", "/mnt/session_data", "/home/jovyan"]), [])), "")
    if not gtf:
        try:
            gtf = "/home/jovyan/gencode.v19.annotation.gtf.gz"
            print("downloading", GTF_URL); urllib.request.urlretrieve(GTF_URL, gtf)
        except Exception as e:
            print("could not download GTF:", e); gtf = ""
    if gtf:
        lengths = gtf_gene_lengths(gtf); lengths.to_csv(len_csv, header=["length_bp"])

In [ ]:
def with_names(mat):
    o = mat.copy(); o.insert(0, "Gene_Name", o.index.map(gene_names)); return o

cc = counts[list(final_meta["sample_id"])]                       # one per patient
ce = counts[list(elig_meta["sample_id"])]                        # all eligible
with_names(cc).to_csv(os.path.join(OUT_DIR, "rnaseq_counts_raw.csv"))
with_names(ce).to_csv(os.path.join(OUT_DIR, "rnaseq_counts_raw_eligible.csv"))
with_names(counts[list(rmeta.loc[rmeta["has_counts"], "sample_id"])]).to_csv(
    os.path.join(OUT_DIR, "rnaseq_counts_all_samples.csv.gz"), compression="gzip")

if lengths is None:
    print("\nNo gene lengths available: raw counts written, RPKM skipped. "
          "Put a .gtf(.gz) on the mount and set GTF_PATH.")
else:
    versioned = cc.index.isin(lengths.index).mean() > 0.5
    lkp, key = lengths.copy(), cc.index
    if not versioned:
        lkp.index = lkp.index.str.split(".").str[0]; lkp = lkp[~lkp.index.duplicated()]
        key = cc.index.str.split(".").str[0]
    L = pd.Series(lkp.reindex(key).values / 1000.0, index=cc.index)
    print(f"\ngene length found for {L.notna().mean():.1%} of {len(L)} genes")
    lib = cc.sum()                                   # full library size, before dropping genes
    ok = L.notna() & (L > 0)
    c2 = cc[ok]
    rpkm = c2.div(L[ok], axis=0).div(lib / 1e6, axis=1)
    rate = c2.div(L[ok], axis=0); tpm = rate.div(rate.sum(), axis=1) * 1e6
    for nm, mat in (("rpkm", rpkm), ("tpm", tpm)):
        with_names(mat).to_csv(os.path.join(OUT_DIR, f"rnaseq_{nm}.csv"))

### Checkpoint 10
Left: TPM column sums should all be 1,000,000. Right: per-sample log TPM curves should overlay closely; one far off is an outlier.

In [ ]:
if lengths is not None:
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].hist(tpm.sum(), bins=20, color="#4C78A8"); ax[0].set_title("TPM column sums (should be 1e6)")
    for s in tpm.columns[:60]:
        ax[1].hist(np.log2(tpm[s] + 1), bins=60, histtype="step", alpha=.4, density=True)
    ax[1].set_title("log2(TPM+1) per sample (first 60)")
    plt.tight_layout(); plt.show()
else:
    print("no gene lengths, nothing to plot")

---
# Part 11 · Write the sample sheets and audit

In [ ]:
cols = ["sample_id", "patient_key", "enrollment_id_rna", "initials_rna", "meta_join",
        "demographics_trusted", "sample_source", "treatment", "tx_status", "diagnosis_rna",
        "dx_clean", "diagnosis1", "diagnosis2", "study_status", "sex", "race", "ethnicity",
        "age_yrs", "age_date_source", "sequencing_facility", "sequencing_cohort",
        "facility", "batch", "total_counts", "source_file", "source_sheet"]
final_meta[cols].to_csv(os.path.join(OUT_DIR, "rnaseq_samples.csv"), index=False)
elig_meta.assign(one_per_patient=elig_meta["kept"])[cols + ["one_per_patient"]].to_csv(
    os.path.join(OUT_DIR, "rnaseq_samples_eligible.csv"), index=False)
final_meta.loc[~final_meta["demographics_trusted"],
               ["sample_id", "enrollment_id_rna", "initials_rna", "meta_join", "dx_clean", "source_file"]
               ].to_csv(os.path.join(OUT_DIR, "unmatched_or_weak_metadata.csv"), index=False)
rmeta.drop(columns=["_dob", "_date", "_md_enroll", "meta_row"]).to_csv(
    os.path.join(OUT_DIR, "rnaseq_samples_ALL_with_flags.csv"), index=False)
bt_elig.to_csv(os.path.join(OUT_DIR, "batch_by_diagnosis_eligible.csv"))

log = ["# RNA-seq processing log (notebook 03)",
       f"Generated {datetime.datetime.now():%Y-%m-%d %H:%M}.\n", "## Units read", "```", units_df.to_string(index=False), "```", "",
       "## Rules, in order",
       f"0. no counts / library < {MIN_LIBRARY:,.0f} excluded; 1. whole blood only; 2. post-treatment excluded;",
       f"3. explicit PRE required for: {REQUIRE_PRE_FOR_DX or '(nobody)'};",
       f"4. one per patient across facilities: explicit PRE, then '{KEEP_ONE_PER_PATIENT}', then deepest library.",
       "Batch = Sequencing_Facility (NIAMS split by Sequencing_Cohort).", "",
       f"Samples read: {len(rmeta)}; eligible: {int(elig.sum())}; one per patient: {len(final_meta)}.", "",
       "## Fate of every sample", "```", rmeta["status"].value_counts().to_string(), "```", "",
       "## Batch x diagnosis (eligible)", "```", bt_elig.to_string(), "```"]
open(os.path.join(OUT_DIR, "PROCESSING_LOG_notebook03.md"), "w").write("\n".join(log))
print("written to", OUT_DIR, ":")
for f in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, f)
    if os.path.isfile(fp): print(f"  {f:46s} {os.path.getsize(fp)/1e6:8.1f} MB")

k = final_meta
w = k[~k["meta_join"].isin(["enrollment_id", "R_prefix_id"])]
print("\nmetadata join of the kept samples:")
print(pd.crosstab(k["dx_clean"], k["meta_join"]).to_string())
print(f"\nweak / unmatched: {len(w)}")

---
## Done. What to check
1. **Part 4:** do the per-facility sample counts match notebook 02's table?
2. **Checkpoint 8:** are the pre/post calls right? This regex decides cohort membership.
3. **Part 9:** read the batch × diagnosis table. Which batches hold one diagnosis only?
4. `unmatched_or_weak_metadata.csv`: fix IDs at the source and re-run.
5. Part 10: gene-length match rate should be about 95% or more.

**Next:** open **04_NIH_bulk_RNA_seq_batch_PCA_ComBatSeq.ipynb** (kernel: R).